In [ ]:
# 必要なモジュールをインポートする
import sqlite3
import requests
from bs4 import BeautifulSoup
import re
import time
import chardet  # エンコーディング検出用

# データベースの初期化またはアップデート
def init_db():
    conn = sqlite3.connect("hotels.db")  # データベース名前はhotels.db
    cursor = conn.cursor()
    
    # テーブルが存在しない場合は作成
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS hotel_info (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            hotel_name TEXT,
            price TEXT,
            walking_time TEXT
        )
    """)
    
    # `walking_time` カラムが存在しない場合は追加
    try:
        cursor.execute("ALTER TABLE hotel_info ADD COLUMN walking_time TEXT")
    except sqlite3.OperationalError:
        # カラムが既に存在する場合はエラーを無視
        pass

    conn.commit()  # データベースに変更を反映
    conn.close()  # データベースを閉じる

# データを挿入
def insert_data(hotel_name, price, walking_time):
    conn = sqlite3.connect("hotels.db")
    cursor = conn.cursor()
    cursor.execute("INSERT INTO hotel_info (hotel_name, price, walking_time) VALUES (?, ?, ?)", (hotel_name, price, walking_time))
    conn.commit()
    conn.close()

# 駅から徒歩時間を抽出する関数
def extract_walking_time(catch_phrase):
    match = re.search(r"徒歩(\d+)分", catch_phrase)
    if match:
        return f"{match.group(1)}分"
    else:
        return "情報なし"

# 指定したURLからスクレイピング
def scrape_url(url):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }
    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        # レスポンスのエンコーディングを自動検出
        detected_encoding = chardet.detect(response.content)["encoding"]

        # BeautifulSoupにエンコーディングを明示
        soup = BeautifulSoup(response.content, "html.parser", from_encoding=detected_encoding)
        hotel_items = soup.find_all("li", class_="p-yadoCassette p-searchResultItem")

        for item in hotel_items:
            try:
                # ホテル名
                hotel_name = item.find("h2", class_="p-searchResultItem__facilityName").text.strip()

                # 値段
                price = item.find("span", class_="p-searchResultItem__perPersonPrice").text.strip()

                # 徒歩時間をキャッチフレーズから抽出
                catch_phrase_tag = item.find("p", class_="p-searchResultItem__catchPhrase")
                if catch_phrase_tag:
                    catch_phrase = catch_phrase_tag.text.strip()
                    walking_time = extract_walking_time(catch_phrase)
                else:
                    walking_time = "情報なし"

                print(f"ホテル名: {hotel_name}, 値段: {price}, 徒歩時間: {walking_time}")

                # データベースに挿入
                insert_data(hotel_name, price, walking_time)

                # サーバー負荷を避けるためのスリープ
                time.sleep(1)
            except AttributeError:
                # 必要な要素が見つからない場合にスキップ
                continue
    else:
        print(f"Failed to fetch the page. Status code: {response.status_code}")

# メイン処理
if __name__ == "__main__":
    init_db()  # データベースの初期化またはアップデート

    # スクレイピングするURLリスト
    urls = [
        "https://www.jalan.net/uw/uwp2011/uww2011search.do?actionId=G&keyword=%82%DD%82%C8%82%C6%82%DD%82%E7%82%A2&dateUndecided=1&stayYear=2025&stayMonth=01&stayDay=14&adultNum=1&minPrice=0&maxPrice=999999&distCd=06&rootCd=7701&dispStartIndex=0&activeSort=0&screenId=UWW2011",
        "https://www.jalan.net/uw/uwp2011/uww2011search.do?actionId=G&keyword=%82%DD%82%C8%82%C6%82%DD%82%E7%82%A2&dateUndecided=1&stayYear=2025&stayMonth=01&stayDay=14&adultNum=1&minPrice=0&maxPrice=999999&distCd=06&rootCd=7701&dispStartIndex=30&activeSort=0&screenId=UWW2011",
        "https://www.jalan.net/uw/uwp2011/uww2011search.do?actionId=G&keyword=%82%DD%82%C8%82%C6%82%DD%82%E7%82%A2&dateUndecided=1&stayYear=2025&stayMonth=01&stayDay=14&adultNum=1&minPrice=0&maxPrice=999999&distCd=06&rootCd=7701&dispStartIndex=60&activeSort=0&screenId=UWW2011",
        "https://www.jalan.net/uw/uwp2011/uww2011search.do?actionId=G&keyword=%82%DD%82%C8%82%C6%82%DD%82%E7%82%A2&dateUndecided=1&stayYear=2025&stayMonth=01&stayDay=14&adultNum=1&minPrice=0&maxPrice=999999&distCd=06&rootCd=7701&dispStartIndex=90&activeSort=0&screenId=UWW2011"
    ]

    # 各URLを順にスクレイピング
    for url in urls:
        scrape_url(url)
